In [1]:
# Install required packages
!pip install gymnasium numpy matplotlib

   ---------------------------------------- 0.0/953.9 kB ? eta -:--:--
   --------------------- ------------------ 524.3/953.9 kB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 953.9/953.9 kB 5.5 MB/s  0:00:00

   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   ---------------------------------------- 3/3 [gymnasium]




[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import time

print("Gymnasium version:", gym.__version__)

Gymnasium version: 1.3.0


In [4]:
# Create the environment
env = gym.make('CliffWalking-v1')

# Get environment properties
n_states = env.observation_space.n
n_actions = env.action_space.n

print(f"Number of states: {n_states}")
print(f"Number of actions: {n_actions}")
print(f"Action mapping: 0=Up, 1=Right, 2=Down, 3=Left")

# Display the transition model for a sample state
print("\nTransition model for state 36 (start position):")
for prob, next_state, reward, terminated in env.unwrapped.P[36][1]:  # action 1 = Right
    print(f"  prob={prob:.2f}, next_state={next_state}, reward={reward}, terminated={terminated}")

Number of states: 48
Number of actions: 4
Action mapping: 0=Up, 1=Right, 2=Down, 3=Left

Transition model for state 36 (start position):
  prob=1.00, next_state=36, reward=-100, terminated=False


In [5]:
def visualize_cliff_world():
    """Visualize the Cliff Walking grid layout."""
    grid = np.full((4, 12), '.')
    grid[3, 0] = 'S'  # Start
    grid[3, 11] = 'G'  # Goal
    grid[3, 1:11] = 'C'  # Cliff
    
    print("Cliff Walking Environment Layout:")
    print("  S = Start, G = Goal, C = Cliff, . = Safe path")
    print("  " + "-" * 25)
    for row in range(4):
        row_str = ""
        for col in range(12):
            if row == 3 and 1 <= col <= 10:
                row_str += " C "
            elif row == 3 and col == 0:
                row_str += " S "
            elif row == 3 and col == 11:
                row_str += " G "
            else:
                row_str += " . "
        print(f"  {row_str}")
        print(f"  " + "-" * 35)

visualize_cliff_world()

Cliff Walking Environment Layout:
  S = Start, G = Goal, C = Cliff, . = Safe path
  -------------------------
   .  .  .  .  .  .  .  .  .  .  .  . 
  -----------------------------------
   .  .  .  .  .  .  .  .  .  .  .  . 
  -----------------------------------
   .  .  .  .  .  .  .  .  .  .  .  . 
  -----------------------------------
   S  C  C  C  C  C  C  C  C  C  C  G 
  -----------------------------------


In [6]:
class PolicyIterationAgent:
    """
    Policy Iteration agent for discrete MDPs.
    
    Implements the classic dynamic programming algorithm that alternates between
    policy evaluation and policy improvement until convergence to the optimal policy.
    """
    
    def __init__(self, env, gamma=0.99, theta=1e-8, max_eval_iterations=1000):
        """
        Initialize the Policy Iteration agent.
        
        Parameters:
        -----------
        env : gymnasium.Env
            The environment (must have P attribute with transition probabilities)
        gamma : float
            Discount factor (0 <= gamma <= 1)
        theta : float
            Convergence threshold for policy evaluation
        max_eval_iterations : int
            Maximum iterations for policy evaluation loop
        """
        self.env = env
        self.gamma = gamma
        self.theta = theta
        self.max_eval_iterations = max_eval_iterations
        
        self.n_states = env.observation_space.n
        self.n_actions = env.action_space.n
        
        # Get transition model from environment
        # P[state][action] = list of (prob, next_state, reward, terminated)
        self.P = env.unwrapped.P
        
        # Initialize policy uniformly random
        self.policy = np.random.choice(self.n_actions, size=self.n_states)
        
        # Initialize value function
        self.V = np.zeros(self.n_states)
        
        # Track convergence metrics
        self.evaluation_iterations = []
        self.improvement_steps = 0
        
    def policy_evaluation(self):
        """
        Evaluate the current policy by solving Bellman expectation equation.
        
        Iteratively applies the Bellman operator until convergence or
        maximum iterations reached.
        
        Returns:
        --------
        V : np.ndarray
            State-value function for the current policy
        iterations : int
            Number of iterations performed
        """
        V = np.zeros(self.n_states)
        
        for iteration in range(self.max_eval_iterations):
            delta = 0
            V_prev = V.copy()
            
            # Iterate over all states
            for s in range(self.n_states):
                action = self.policy[s]
                
                # Compute value for this state under current policy
                v_new = 0
                for prob, next_state, reward, terminated in self.P[s][action]:
                    v_new += prob * (reward + self.gamma * V_prev[next_state])
                
                V[s] = v_new
                delta = max(delta, abs(V_prev[s] - V[s]))
            
            # Check for convergence
            if delta < self.theta:
                return V, iteration + 1
        
        return V, self.max_eval_iterations
    
    def policy_improvement(self):
        """
        Improve the policy by acting greedily with respect to current value function.
        
        For each state, computes Q-values for all actions and selects the best one.
        
        Returns:
        --------
        new_policy : np.ndarray
            Greedy policy derived from current value function
        policy_stable : bool
            Whether the policy changed (False if improved, True if stable)
        """
        new_policy = np.zeros(self.n_states, dtype=int)
        policy_stable = True
        
        for s in range(self.n_states):
            # Compute Q-values for all actions
            q_values = np.zeros(self.n_actions)
            
            for a in range(self.n_actions):
                q_value = 0
                for prob, next_state, reward, terminated in self.P[s][a]:
                    q_value += prob * (reward + self.gamma * self.V[next_state])
                q_values[a] = q_value
            
            # Select best action
            best_action = np.argmax(q_values)
            new_policy[s] = best_action
            
            # Check if policy changed
            if best_action != self.policy[s]:
                policy_stable = False
        
        return new_policy, policy_stable
    
    def train(self, max_iterations=100, verbose=True):
        """
        Run the policy iteration algorithm until convergence.
        
        Parameters:
        -----------
        max_iterations : int
            Maximum number of policy improvement steps
        verbose : bool
            Whether to print progress information
            
        Returns:
        --------
        policy : np.ndarray
            The optimal policy found
        V : np.ndarray
            The optimal value function
        metrics : dict
            Training metrics (evaluation steps per iteration, total time)
        """
        start_time = time.time()
        
        for i in range(max_iterations):
            # Policy Evaluation
            self.V, eval_iters = self.policy_evaluation()
            self.evaluation_iterations.append(eval_iters)
            
            # Policy Improvement
            new_policy, policy_stable = self.policy_improvement()
            self.improvement_steps = i + 1
            
            if verbose:
                print(f"Iteration {i+1}: Policy eval converged in {eval_iters} iterations")
                print(f"             Policy changed: {not policy_stable}")
            
            if policy_stable:
                if verbose:
                    print(f"\n✓ Policy converged after {i+1} improvement steps!")
                break
            
            self.policy = new_policy
        
        total_time = time.time() - start_time
        
        metrics = {
            'improvement_steps': self.improvement_steps,
            'evaluation_iterations': self.evaluation_iterations,
            'total_time': total_time
        }
        
        return self.policy, self.V, metrics

In [7]:
# Create agent with discount factor gamma=0.99
agent = PolicyIterationAgent(env, gamma=0.99, theta=1e-8)

# Train the agent
optimal_policy, optimal_value, metrics = agent.train(max_iterations=100, verbose=True)

Iteration 1: Policy eval converged in 1000 iterations
             Policy changed: True
Iteration 2: Policy eval converged in 1000 iterations
             Policy changed: False

✓ Policy converged after 2 improvement steps!


In [8]:
def visualize_policy(policy, n_rows=4, n_cols=12):
    """
    Visualize the policy as arrows on the grid.
    
    Parameters:
    -----------
    policy : np.ndarray
        Array of actions (0-3) for each state
    n_rows, n_cols : int
        Grid dimensions
    """
    action_symbols = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    
    print("\nOptimal Policy (showing best action in each cell):")
    print("  " + "=" * 49)
    
    for row in range(n_rows):
        row_str = ""
        for col in range(n_cols):
            state = row * n_cols + col
            
            # Check if this is start, goal, or cliff
            if row == 3 and col == 0:
                row_str += "  S  "
            elif row == 3 and col == 11:
                row_str += "  G  "
            elif row == 3 and 1 <= col <= 10:
                row_str += "  C  "  # Cliff (fall)
            else:
                action = policy[state]
                row_str += f"  {action_symbols[action]}  "
        
        print(f"  {row_str}")
        print("  " + "-" * 49)

# Visualize the learned optimal policy
visualize_policy(optimal_policy)


Optimal Policy (showing best action in each cell):
    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑  
  -------------------------------------------------
    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑  
  -------------------------------------------------
    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑  
  -------------------------------------------------
    S    C    C    C    C    C    C    C    C    C    C    G  
  -------------------------------------------------


In [9]:
def analyze_policy(policy, n_rows=4, n_cols=12):
    """Analyze and explain the optimal policy."""
    action_names = {0: "UP", 1: "RIGHT", 2: "DOWN", 3: "LEFT"}
    
    # Count action distribution
    action_counts = defaultdict(int)
    for state, action in enumerate(policy):
        row = state // n_cols
        col = state % n_cols
        # Exclude cliff cells (these are terminal states)
        if not (row == 3 and 1 <= col <= 10):
            action_counts[action] += 1
    
    print("\n=== Policy Analysis ===")
    print(f"Total improvement steps: {metrics['improvement_steps']}")
    print(f"Total training time: {metrics['total_time']:.4f} seconds")
    print(f"Average evaluation iterations per improvement: {np.mean(metrics['evaluation_iterations']):.1f}")
    
    print("\nAction distribution (excluding cliff cells):")
    for action, count in sorted(action_counts.items()):
        print(f"  {action_names[action]}: {count} states")
    
    # Check if policy follows the safe path along the top
    top_row_actions = [policy[col] for col in range(n_cols)]
    if all(a == 1 for a in top_row_actions):  # All RIGHT in top row
        print("\n✓ Optimal policy follows the SAFE PATH along the top row (moving RIGHT)")
        print("  Reason: The agent stays away from the cliff to avoid -100 penalty,")
        print("  even though this results in a longer path (more steps with -1 reward).")
    else:
        print("\n✗ Policy does not follow simple safe path pattern")
    
    # Check bottom row behavior
    bottom_actions = [policy[36 + col] for col in range(n_cols)]  # state 36 = row3, col0
    print(f"\nBottom row (safe cells only):")
    for col, action in enumerate(bottom_actions):
        if col == 0:
            print(f"  Start (col 0): {action_names[action]} (moving away from cliff)")
        elif col == 11:
            print(f"  Goal (col 11): Goal state (terminal)")

analyze_policy(optimal_policy)


=== Policy Analysis ===
Total improvement steps: 2
Total training time: 0.1283 seconds
Average evaluation iterations per improvement: 1000.0

Action distribution (excluding cliff cells):
  UP: 38 states

✗ Policy does not follow simple safe path pattern

Bottom row (safe cells only):
  Start (col 0): UP (moving away from cliff)
  Goal (col 11): Goal state (terminal)


In [10]:
def evaluate_policy(env, policy, n_episodes=100, max_steps=200):
    """
    Evaluate a policy by running multiple episodes.
    
    Parameters:
    -----------
    env : gymnasium.Env
        The environment
    policy : np.ndarray
        Policy to evaluate (mapping state -> action)
    n_episodes : int
        Number of episodes to run
    max_steps : int
        Maximum steps per episode
        
    Returns:
    --------
    results : dict
        Dictionary containing rewards, steps, success rates
    """
    total_rewards = []
    episode_lengths = []
    successes = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        episode_reward = 0
        steps = 0
        
        for step in range(max_steps):
            action = policy[state]
            next_state, reward, terminated, truncated, _ = env.step(action)
            
            episode_reward += reward
            steps += 1
            state = next_state
            
            if terminated or truncated:
                # Success if reached goal (not cliff)
                success = (reward == 0)  # Goal reward is 0 (not -100)
                successes.append(success)
                break
        
        total_rewards.append(episode_reward)
        episode_lengths.append(steps)
    
    return {
        'mean_reward': np.mean(total_rewards),
        'std_reward': np.std(total_rewards),
        'mean_length': np.mean(episode_lengths),
        'std_length': np.std(episode_lengths),
        'success_rate': np.mean(successes),
        'all_rewards': total_rewards,
        'all_lengths': episode_lengths
    }

# Evaluate the optimal policy
print("Evaluating optimal policy...")
eval_results = evaluate_policy(env, optimal_policy, n_episodes=1000)

print(f"\n=== Performance Metrics (1000 episodes) ===")
print(f"Success rate: {eval_results['success_rate']*100:.1f}%")
print(f"Mean episode reward: {eval_results['mean_reward']:.2f} ± {eval_results['std_reward']:.2f}")
print(f"Mean episode length: {eval_results['mean_length']:.1f} ± {eval_results['std_length']:.1f} steps")

Evaluating optimal policy...

=== Performance Metrics (1000 episodes) ===
Success rate: nan%
Mean episode reward: -200.00 ± 0.00
Mean episode length: 200.0 ± 0.0 steps


c:\Users\Andrey\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Andrey\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
